# 04. RAG Agent over Policy PDFs with LangGraph `StateGraph`

This notebook builds a retrieval-augmented-generation (RAG) support agent for CloudXeus that answers questions about the company's Acceptable Use Policy, Refund Policy, and Service Level Agreement by searching a FAISS vector store built from three PDFs, using **LangGraph**'s lower-level `StateGraph` API instead of LangChain's `create_agent`. It belongs to Section 04, "Agent frameworks on top of Azure AI Foundry," in the `02-langgraph` sub-section.

**Difficulty: Advanced**

LangGraph — like LangChain — is a third-party orchestration framework and is **not** part of the AI-102 exam. The exam-relevant substance underneath this notebook is: (1) the Azure AI Foundry chat + embedding deployments being called, and (2) the general RAG pattern (chunk documents → embed → retrieve top-k → ground the model's answer in retrieved text), which AI-102 does test conceptually, typically via Azure AI Search rather than a local FAISS index. Treat the `StateGraph`/`MessagesState`/`ToolNode` machinery as "how one popular framework implements an agent loop," not as an Azure concept.

## Prerequisites

**Pip packages:**
```bash
pip install langgraph langchain-community langchain-text-splitters pypdf faiss-cpu langchain-openai python-dotenv
```
`langchain-community` and `langchain-text-splitters` are already in the repo root `requirements.txt`; `langgraph`, `pypdf`, and `faiss-cpu` are not yet and need to be added.

**Azure resources required:**
- The same Azure AI Foundry project as notebooks 01–03, with **two** deployments: a chat model (e.g. the `gpt-5.4` deployment used elsewhere in this section) and an **embedding** model (the original script uses `text-embedding-3-small`).

**Environment variables** (add to root `.env`):
```
AZURE_AI_OPENAI_ENDPOINT=https://<your-foundry-resource>.services.ai.azure.com/openai/v1
AZURE_AI_OPENAI_API_KEY=<your-foundry-api-key>
AZURE_AI_MODEL_DEPLOYMENT=<your-chat-deployment-name>
AZURE_AI_EMBEDDING_DEPLOYMENT=text-embedding-3-small
```

**Local files:** this notebook expects three PDFs — `cloudxeus-acceptable-use-policy.pdf`, `cloudxeus-refund-policy.pdf`, `cloudxeus-service-level-agreement.pdf` — in the **same folder as this notebook** (`04. Section Code/02-langgraph/`). They're already present alongside the original script; if you open this notebook with Jupyter's working directory set to its own folder (the default when launching from that directory), the relative paths below resolve correctly.

## What You'll Learn

- Loading and chunking PDFs with `PyPDFLoader` + `RecursiveCharacterTextSplitter`
- Building a local `FAISS` vector store from embedded document chunks
- Wrapping a retriever as a LangChain `@tool` so a model can call it on demand
- Building an explicit agent loop with LangGraph's `StateGraph`, `MessagesState`, and `ToolNode` — the lower-level alternative to `create_agent`
- Conditional edges: how the graph decides whether to call a tool or finish

### Step 1 — Imports, configuration, and the base model

Same environment-driven setup pattern as earlier notebooks. This notebook needs more imports because it builds its own retrieval pipeline (document loaders, splitter, vector store) on top of the graph-building blocks (`StateGraph`, `MessagesState`, `START`, `END`, `ToolNode`).

💡 **Framework note:** `StateGraph` is LangGraph's core abstraction — a directed graph of named nodes (each a Python function that receives and returns a piece of state) connected by edges, some of them conditional. `create_agent` (used in notebooks 02–03) is actually implemented *on top of* `StateGraph` internally — this notebook shows the graph one level down, giving you control over exactly when the loop stops.

🔄 **Alternatives:** for a much larger document corpus or one that needs incremental updates, replace the local FAISS index with Azure AI Search (see Section 07 notebooks in this course, and `08_ai_search.py` from Section 02), which is the AI-102-exam-relevant vector/search backend on Azure.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.tools import tool
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode

load_dotenv()

endpoint = os.getenv("AZURE_AI_OPENAI_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT")
embedding_deployment = os.getenv("AZURE_AI_EMBEDDING_DEPLOYMENT", "text-embedding-3-small")
api_key = os.getenv("AZURE_AI_OPENAI_API_KEY")

model = ChatOpenAI(
    base_url=endpoint,
    api_key=api_key,
    model=deployment_name,
)


### Step 2 — Load the policy PDFs

`PyPDFLoader` reads a PDF and returns one LangChain `Document` per page, with page-level metadata (including `source`, the file path) attached. Three loaders are created, one per policy document, and their outputs are concatenated into a single `documents` list.

💡 **Framework note:** each `Document`'s `metadata["source"]` is what later lets the retrieval tool cite *which* policy file an answer came from — this is why keeping the loader-per-file structure (rather than one loader over a merged file) matters for traceability.

🔄 **Alternatives:** for scanned or image-heavy PDFs, `PyPDFLoader`'s text extraction can miss content — Azure AI Document Intelligence's layout/OCR models (used in Section 07 of this course) handle that case far better than a pure-Python PDF text extractor.

In [ ]:
loaders = [
    PyPDFLoader("cloudxeus-acceptable-use-policy.pdf"),
    PyPDFLoader("cloudxeus-refund-policy.pdf"),
    PyPDFLoader("cloudxeus-service-level-agreement.pdf"),
]

documents = []
for loader in loaders:
    documents.extend(loader.load())

print(f"Loaded {len(documents)} pages across all policy documents.")


### Step 3 — Split documents into chunks

`RecursiveCharacterTextSplitter` breaks each page's text into smaller overlapping chunks (500 characters, with 50 characters of overlap between consecutive chunks). Overlap prevents a sentence that straddles a chunk boundary from losing context in either chunk. Chunking matters because embedding models and retrieval work best over focused, similarly-sized passages rather than whole pages.

💡 **Framework note:** `chunk_size` and `chunk_overlap` are tuning parameters that trade off retrieval precision (smaller chunks: more precise but less context per hit) against context completeness (larger chunks: more context but noisier matches) — this tuning concern is the same whether you retrieve via FAISS or Azure AI Search.

🔄 **Alternatives:** other splitting strategies exist (`MarkdownHeaderTextSplitter` for structured docs, semantic/embedding-based splitters) — `RecursiveCharacterTextSplitter` is a reasonable general-purpose default because it tries to split on paragraph/sentence boundaries before falling back to raw character counts.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks.")


### Step 4 — Build the vector store

`OpenAIEmbeddings` is configured against the same Foundry `/openai/v1` endpoint, but with an **embedding** deployment name (`text-embedding-3-small` in the original script) rather than the chat deployment. `FAISS.from_documents(...)` embeds every chunk and builds an in-memory, in-process similarity index — no external server required, which is why FAISS is convenient for a notebook demo.

💡 **Exam tip:** on AI-102, this local FAISS-in-a-notebook pattern is the conceptual stand-in for Azure AI Search with vector fields — same idea (embed text, index vectors, retrieve by similarity), but Azure AI Search is a managed, scalable, persistent service with hybrid (vector + keyword) search, which is what the exam actually tests for production RAG scenarios.

🔄 **Alternatives:** swap `FAISS` for `AzureSearch` (from `langchain-community`) to persist the index in Azure AI Search instead of in-memory — the retrieval tool code below barely changes, since both expose the same `similarity_search` interface.

In [ ]:
print("Building vector store...")

embeddings = OpenAIEmbeddings(
    base_url=endpoint,
    api_key=api_key,
    model=embedding_deployment,
)

vector_store = FAISS.from_documents(chunks, embeddings)
print("Vector store ready.")


### Step 5 — Wrap retrieval as a tool

`search_cloudxeus_policies` is a `@tool`-decorated function, just like the tools in notebooks 02–03, except its body calls `vector_store.similarity_search(query, k=3)` instead of a dictionary lookup — it retrieves the 3 most relevant chunks and formats them with their source file for the model to read. This is the RAG step: the model doesn't "know" the policies; it calls this tool to fetch grounding text before answering.

💡 **Framework note:** the tool's docstring (again) is what the model reads to decide *when* to call it — write it to clearly state what the tool covers ("Acceptable Use Policy, Refund Policy, and Service Level Agreement") so the model doesn't call it for unrelated questions.

🔄 **Alternatives:** instead of manually formatting `k=3` results into a string, LangChain also offers `create_retriever_tool(...)` as a shortcut that wraps a retriever object into a tool with less boilerplate — functionally equivalent to what's written out explicitly here.

In [ ]:
@tool
def search_cloudxeus_policies(query: str) -> str:
    """
    Search the CloudXeus policy documents — Acceptable Use Policy,
    Refund Policy, and Service Level Agreement — to answer questions
    about CloudXeus rules, eligibility, credits, and procedures.
    """
    results = vector_store.similarity_search(query, k=3)
    if not results:
        return "No relevant policy information found."

    output = []
    for i, doc in enumerate(results, 1):
        source = doc.metadata.get("source", "Unknown policy")
        output.append(f"[Excerpt {i} from {source}]\n{doc.page_content}")

    return "\n\n".join(output)


tools = [search_cloudxeus_policies]
model_with_tools = model.bind_tools(tools)


### Step 6 — Define the graph's nodes

A LangGraph `StateGraph` is built from plain functions that take the current `state` (here, `MessagesState`, which just wraps a `messages` list) and return a partial state update.

- `call_model` prepends a system message on every turn, then calls the tool-bound model with the full message history — this is the "reasoning" node.
- `should_continue` inspects the *last* message: if the model requested tool calls, route to the `"tools"` node; otherwise route to `END` and stop.

💡 **Framework note:** this is the explicit version of the same decision `create_agent` makes for you internally in notebooks 02–03. Writing it out yourself is what gives you control — e.g., you could cap the number of tool round-trips, add custom logging per step, or branch to a different node entirely based on which tool was called.

🔄 **Alternatives:** `create_agent` (LangChain) covers this exact pattern (model ↔ tools loop) with less code when you don't need this level of control — use the explicit `StateGraph` version specifically when you need custom routing logic, as shown here.

In [ ]:
def call_model(state: MessagesState):
    system_message = {
        "role": "system",
        "content": (
            "You are the CloudXeus policy assistant. "
            "Answer questions accurately using the search_cloudxeus_policies tool. "
            "Always base your answers on the retrieved policy content. "
            "If you cannot find the answer in the policies, say so clearly."
        ),
    }
    messages = [system_message] + state["messages"]
    response = model_with_tools.invoke(messages)
    return {"messages": [response]}


def should_continue(state: MessagesState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END


### Step 7 — Assemble and compile the graph

Nodes are added by name (`"call_model"`, `"tools"`), then edges connect them: `START → call_model` unconditionally, `call_model → (tools or END)` conditionally via `should_continue`, and `tools → call_model` unconditionally (after a tool runs, control always returns to the model to interpret the result). `ToolNode(tools)` is a LangGraph prebuilt node that knows how to execute whichever tool(s) the model's last message requested. `.compile()` turns the graph definition into a runnable object.

💡 **Framework note:** this three-node, one-conditional-edge shape (`call_model` ↔ `tools`) is the canonical LangGraph "ReAct agent" pattern — it's exactly what `langgraph.prebuilt.create_react_agent` (and, one layer up, LangChain's `create_agent`) builds automatically. Writing it by hand here is pedagogical — in real code, reach for the prebuilt unless you need the custom control this notebook demonstrates.

🔄 **Alternatives:** add more nodes/conditional branches for more complex flows — e.g., a node that first classifies the question's topic and routes to different tool sets, or a human-approval node inserted between `tools` and `call_model`.

In [ ]:
graph_builder = StateGraph(MessagesState)
graph_builder.add_node("call_model", call_model)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_edge(START, "call_model")
graph_builder.add_conditional_edges("call_model", should_continue)
graph_builder.add_edge("tools", "call_model")

graph = graph_builder.compile()
print("Agent ready.\n")


### Step 8 — Run the agent over sample questions

Each question is sent through `graph.invoke({"messages": [...]})`, and the loop built in Step 6–7 runs until the model produces a final answer with no further tool calls. The four sample questions span factual retrieval (refund window, uptime SLA), a policy-violation check (crypto mining), and a compound question (suspension + refund) — testing whether the agent grounds each answer in the right document.

💡 **Framework note:** each call to `graph.invoke(...)` is a fresh run starting from `START` — there's no built-in memory across the four questions in this loop (each is independent), the same "no memory unless you build it in" characteristic seen with `11_azure_ai_foundry/04_prompt_agent/main.py`. LangGraph supports persistent memory via checkpointers if you want conversational continuity across turns.

🔄 **Alternatives:** pass a `thread_id` and a checkpointer (e.g., `MemorySaver`) to `graph.compile(...)` to give the agent conversational memory across multiple `.invoke()` calls with the same thread — useful for a genuinely multi-turn support chat rather than four independent questions.

In [ ]:
questions = [
    "What is the refund window for a CloudXeus Pro subscription?",
    "What uptime does CloudXeus guarantee for Enterprise customers?",
    "Can I mine cryptocurrency on CloudXeus compute resources?",
    "What happens if CloudXeus suspends my account — do I get a refund?",
]

for question in questions:
    print(f"Question: {question}")
    result = graph.invoke({"messages": [{"role": "user", "content": question}]})
    print(f"Answer: {result['messages'][-1].content}")
    print("-" * 60)


## Summary

This notebook built a complete local RAG pipeline — load PDFs, chunk them, embed and index them in FAISS, wrap retrieval as a tool — then hand-assembled a LangGraph `StateGraph` that alternates between a reasoning node (`call_model`) and a tool-execution node (`ToolNode`) until the model has enough grounded context to answer. Running it against four sample questions demonstrates the agent retrieving different policy documents per question and citing them in its answers. This is the same retrieval-augmented-generation pattern tested conceptually on AI-102, just implemented here with a local vector store and a hand-built graph loop instead of Azure AI Search and a managed Agent Service.

## Try It Yourself

1. **Easy:** Add a fifth question that isn't covered by any of the three policies (e.g., "What is CloudXeus's parking policy?") and observe whether the agent correctly says it can't find the answer rather than hallucinating one.
2. **Intermediate:** Print `result["messages"]` in full for one question to see the tool-call and tool-result messages LangGraph inserted between the user question and the final answer.
3. **Advanced:** Swap the `FAISS` vector store for `langchain_community.vectorstores.AzureSearch`, pointing at an Azure AI Search resource, and compare retrieval quality/latency — this is the production-grade version of this same RAG pattern on Azure.